# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 7 — Structural baselines for the pre-training attribution signal

A randomly initialised graph network concentrates Integrated Gradients on toxicophore atoms
at $1.116\times$ chance. Whether that reflects something about message passing, or merely
tracks a simple property of the atoms themselves, can be settled without training anything.

This notebook ranks atoms by elementary structural descriptors and scores the result with
the same overlap measure used throughout, then regresses the per-molecule untrained signal
on properties of the molecule. Two further checks that need no model are included: the
overlap between the partitions used for the benchmark, and the sensitivity of the agreement
measure to individual entries in the alert library.

**Contents**

1. Environment
2. Evaluation set and scoring
3. Structural rankings
4. What predicts the untrained signal
5. Overlap between partitions
6. Sensitivity to the alert library
7. Summary

*Approximate runtime: one hour on CPU. No GPU and no model training.*

## 1. Environment

In [1]:
!pip install -q rdkit
!pip install -q statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 51.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK   = "/content/drive/MyDrive/ames_pipeline"
DATADIR, R, EXPL = f"{WORK}/data", f"{WORK}/results", f"{WORK}/explain"
os.makedirs(R, exist_ok=True)
print("Results:", R)

Mounted at /content/drive
Results: /content/drive/MyDrive/ames_pipeline/results


In [3]:
import numpy as np, pandas as pd, json
from collections import defaultdict
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

SEED, PRIMARY_SPLIT_SEED, K_FRAC = 42, 101, 0.30
N_TIEBREAK = 200          # binary descriptors leave many ties; average over draws

def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi

def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= a:            tr += g
        elif len(tr) + len(va) + len(g) <= b: va += g
        else:                                 te += g
    return d.loc[tr], d.loc[va], d.loc[te]

def make_random_split(d, seed, frac=(0.8, 0.1, 0.1)):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(d.index.values)
    n = len(d); a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    return d.loc[idx[:a]], d.loc[idx[a:b]], d.loc[idx[b:]]

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}") for f in ["train.csv","valid.csv","test.csv"]],
               ignore_index=True)[["Drug_ID","smiles","label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)
_, _, df_test = [x.reset_index(drop=True) for x in make_scaffold_split(df, PRIMARY_SPLIT_SEED)]
print(f"{len(df)} compounds, primary test partition {len(df_test)}")

7255 compounds, primary test partition 726


## 2. Evaluation set and scoring

The evaluation set and the agreement measure are those used in the attribution analysis: the
observed overlap between the highest-ranked 30\% of atoms and the toxicophore atoms, divided
by the chance overlap, aggregated as a ratio of means. Ties are broken at random. Because
several of the descriptors below are binary, and therefore leave large numbers of ties, each
ranking is scored over 200 independent tie-breaks and the mean is reported with its spread.

The count of molecules excluded for each reason is recorded separately, since a molecule
matched in its entirety leaves the measure no room to discriminate.

In [4]:
TOXICOPHORES = {
    "aromatic_nitro":         "[a][N+](=O)[O-]",
    "aromatic_amine":         "[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":       "[a][NX2]=[OX1]",
    "alkyl_nitrite":          "[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":            "[NX3][NX2]=[OX1]",
    "epoxide":                "C1OC1",
    "aziridine":              "C1NC1",
    "azide":                  "[NX2]=[NX2+]=[NX1-]",
    "azo_aromatic":           "[a][NX2]=[NX2][a]",
    "aliphatic_halide":       "[CX4][F,Cl,Br,I]",
    "hetero_hetero_bond":     "[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":    "c1ccc2ccccc2c1",
    "aromatic_hydroxylamine": "[a][NX3;H1][OX2;H1]",
    "michael_acceptor":       "[CX3]=[CX3][CX3]=[OX1]",
    "aromatic_azoxy":         "[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":       "[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":     "[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":        "[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
TOX_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}
assert all(p is not None for p in TOX_PATTERNS.values())

def tox_atoms(mol, patterns=None):
    patterns = TOX_PATTERNS if patterns is None else patterns
    out = set()
    for p in patterns.values():
        for m in mol.GetSubstructMatches(p):
            out.update(m)
    return out

EVAL, n_no_alert, n_all_alert = [], 0, 0
for smi, lab in zip(df_test["canonical"], df_test["label"]):
    if lab != 1:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0:
        continue
    t = tox_atoms(mol)
    if not t:
        n_no_alert += 1; continue
    if len(t) == mol.GetNumAtoms():
        n_all_alert += 1; continue
    EVAL.append({"smiles": smi, "mol": mol, "tox": t, "n_atoms": mol.GetNumAtoms()})

CHANCE = np.array([len(e["tox"]) / e["n_atoms"] for e in EVAL])
N_EVAL = len(EVAL)
print(f"Mutagenic test compounds      : {int(df_test.label.sum())}")
print(f"  excluded, no alert matched  : {n_no_alert}")
print(f"  excluded, matched entirely  : {n_all_alert}")
print(f"Evaluation set                : {N_EVAL}")
print(f"Mean chance overlap           : {CHANCE.mean():.4f}")

def overlap(scores_list, k_frac=K_FRAC, tie_seed=0, ev=None, chance=None):
    ev = EVAL if ev is None else ev
    rng = np.random.default_rng(tie_seed)
    out = np.full(len(ev), np.nan)
    for i, (e, sc) in enumerate(zip(ev, scores_list)):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac * n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift(ov, chance=None):
    c = CHANCE if chance is None else chance
    m = ~np.isnan(ov)
    return float(ov[m].mean() / c[m].mean()) if m.sum() else np.nan

ORACLE = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])]) for e in EVAL]
CEILING = lift(overlap(ORACLE, tie_seed=0))
print(f"Attainable maximum lift       : {CEILING:.4f}")

Mutagenic test compounds      : 336
  excluded, no alert matched  : 86
  excluded, matched entirely  : 13
Evaluation set                : 237
Mean chance overlap           : 0.4360
Attainable maximum lift       : 1.9144


## 3. Structural rankings

Each descriptor below is computed from the molecular graph alone. If one of them reproduces
the agreement obtained from a randomly initialised network, then the pre-training signal is
a restatement of that descriptor rather than a property of message passing.

The last two entries test the mechanism proposed in the discussion directly. Rarity counts
how many other atoms in the same molecule share an atom's element, degree and aromaticity;
an atom that is unique in its molecule scores highest. Deviation measures how far an atom's
degree departs from the molecular mean.

In [5]:
HETERO = {"N","O","S","P","F","Cl","Br","I","Se","B","Si"}

def desc_scores(name):
    out = []
    for e in EVAL:
        mol = e["mol"]; ats = list(mol.GetAtoms())
        if name == "degree":
            v = [a.GetDegree() for a in ats]
        elif name == "heteroatom":
            v = [1.0 if a.GetSymbol() in HETERO else 0.0 for a in ats]
        elif name == "hetero_neighbours":
            v = [sum(1 for nb in a.GetNeighbors() if nb.GetSymbol() in HETERO) for a in ats]
        elif name == "aromatic":
            v = [1.0 if a.GetIsAromatic() else 0.0 for a in ats]
        elif name == "in_ring":
            v = [1.0 if a.IsInRing() else 0.0 for a in ats]
        elif name == "abs_formal_charge":
            v = [abs(a.GetFormalCharge()) for a in ats]
        elif name == "hydrogens":
            v = [a.GetTotalNumHs() for a in ats]
        elif name == "hetero_plus_degree":
            v = [(2.0 if a.GetSymbol() in HETERO else 0.0) + 0.25 * a.GetDegree() for a in ats]
        elif name == "hetero_or_neighbour":
            v = [1.0 if (a.GetSymbol() in HETERO or
                         any(nb.GetSymbol() in HETERO for nb in a.GetNeighbors())) else 0.0
                 for a in ats]
        elif name == "rarity":
            key = [(a.GetSymbol(), a.GetDegree(), a.GetIsAromatic()) for a in ats]
            cnt = {k: key.count(k) for k in set(key)}
            v = [1.0 / cnt[k] for k in key]
        elif name == "degree_deviation":
            d = np.array([a.GetDegree() for a in ats], dtype=float)
            v = list(np.abs(d - d.mean()))
        else:
            raise ValueError(name)
        out.append(np.asarray(v, dtype=float))
    return out

DESCRIPTORS = ["degree", "heteroatom", "hetero_neighbours", "aromatic", "in_ring",
               "abs_formal_charge", "hydrogens", "hetero_plus_degree",
               "hetero_or_neighbour", "rarity", "degree_deviation"]

rows = []
for name in DESCRIPTORS:
    sc = desc_scores(name)
    v = np.array([lift(overlap(sc, tie_seed=1000 + j)) for j in range(N_TIEBREAK)])
    rows.append({"ranking": name, "lift_mean": v.mean(), "lift_sd": v.std(ddof=1),
                 "lift_p2_5": np.percentile(v, 2.5), "lift_p97_5": np.percentile(v, 97.5),
                 "pct_of_ceiling": 100 * (v.mean() - 1) / (CEILING - 1)})

# references, recomputed here so the comparison is like for like
rnd = np.array([lift(overlap([np.random.default_rng(9000 + j + 7 * i).random(e["n_atoms"])
                              for i, e in enumerate(EVAL)], tie_seed=9000 + j))
                for j in range(N_TIEBREAK)])
rows.append({"ranking": "random atom order", "lift_mean": rnd.mean(), "lift_sd": rnd.std(ddof=1),
             "lift_p2_5": np.percentile(rnd, 2.5), "lift_p97_5": np.percentile(rnd, 97.5),
             "pct_of_ceiling": 100 * (rnd.mean() - 1) / (CEILING - 1)})
rows.append({"ranking": "perfect ranking (ceiling)", "lift_mean": CEILING, "lift_sd": np.nan,
             "lift_p2_5": np.nan, "lift_p97_5": np.nan, "pct_of_ceiling": 100.0})

struct = pd.DataFrame(rows).sort_values("lift_mean", ascending=False)
struct.to_csv(f"{R}/table29_structural_baselines.csv", index=False)
print(struct.round(4).to_string(index=False))

try:
    n12 = pd.read_csv(f"{R}/table12_untrained_null.csv")
    tg  = pd.read_csv(f"{R}/table13_training_gain.csv").iloc[0]
    u = n12[n12.arch == "gin"]["mean"].iloc[0]
    print(f"\nreference  untrained GIN {u:.4f}   trained GIN "
          f"{tg.trained_lift_mean_over_models:.4f}   chance {rnd.mean():.4f}")
    best = struct.iloc[0]
    if best.ranking not in ("perfect ranking (ceiling)",):
        print(f"strongest structural ranking: {best.ranking} at {best.lift_mean:.4f}, "
              f"{100*(best.lift_mean-1)/(u-1):.0f}% of the untrained signal")
except FileNotFoundError:
    print("\n(reference values not found; run Parts 4 and 5 first)")

                  ranking  lift_mean  lift_sd  lift_p2_5  lift_p97_5  pct_of_ceiling
perfect ranking (ceiling)     1.9144      NaN        NaN         NaN        100.0000
                   degree     1.2135   0.0130     1.1899      1.2386         23.3524
        hetero_neighbours     1.1632   0.0164     1.1317      1.1941         17.8433
         degree_deviation     1.1295   0.0126     1.1060      1.1534         14.1657
       hetero_plus_degree     1.0922   0.0106     1.0724      1.1128         10.0833
        abs_formal_charge     1.0669   0.0225     1.0244      1.1109          7.3161
               heteroatom     1.0593   0.0134     1.0329      1.0837          6.4896
      hetero_or_neighbour     1.0545   0.0187     1.0169      1.0920          5.9604
        random atom order     1.0049   0.0230     0.9566      1.0420          0.5356
                  in_ring     0.9827   0.0184     0.9471      1.0206         -1.8870
                 aromatic     0.9825   0.0157     0.9503      1.0

## 4. What predicts the untrained signal

The rankings above ask whether a descriptor reproduces the aggregate effect. A second and
complementary question is which molecules carry it. The per-molecule agreement of the 500
untrained networks is regressed on properties of the molecule, and the same regression is
run for the trained networks, so that the two can be compared.

In [6]:
import statsmodels.api as sm

def mol_features(e):
    mol = e["mol"]; ats = list(mol.GetAtoms())
    deg = np.array([a.GetDegree() for a in ats], dtype=float)
    return {"alert_coverage": len(e["tox"]) / e["n_atoms"],
            "n_atoms": e["n_atoms"],
            "heteroatom_fraction": np.mean([a.GetSymbol() in HETERO for a in ats]),
            "aromatic_fraction": np.mean([a.GetIsAromatic() for a in ats]),
            "mean_degree": deg.mean(),
            "degree_spread": deg.std()}

X = pd.DataFrame([mol_features(e) for e in EVAL])

def fit(y, label):
    m = ~np.isnan(y)
    model = sm.OLS(y[m], sm.add_constant(X[m])).fit()
    out = {"target": label, "n": int(m.sum()), "r_squared": model.rsquared}
    for k in X.columns:
        out[f"beta_{k}"] = model.params[k]
        out[f"p_{k}"] = model.pvalues[k]
    return out, model

rows = []
paths = {"untrained": f"{EXPL}/untrained_overlaps_gin.npy",
         "trained":   f"{EXPL}/trained_overlaps.npz"}
if os.path.exists(paths["untrained"]):
    un = np.nanmean(np.load(paths["untrained"]), axis=0)
    r, m = fit(un, "untrained GIN, mean overlap")
    rows.append(r); print(m.summary().tables[1])
    print(f"R^2 = {m.rsquared:.3f}\n")
else:
    print("untrained overlaps not found; skipping")

if os.path.exists(paths["trained"]):
    tr = np.nanmean(np.load(paths["trained"])["overlaps"], axis=0)
    r, m = fit(tr, "trained GIN, mean overlap")
    rows.append(r); print(m.summary().tables[1])
    print(f"R^2 = {m.rsquared:.3f}")
else:
    print("trained overlaps not found; skipping")

if rows:
    reg = pd.DataFrame(rows)
    reg.to_csv(f"{R}/table30_signal_regression.csv", index=False)
    print()
    print(reg[["target", "n", "r_squared"]].round(4).to_string(index=False))

                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.0779      0.238     -0.327      0.744      -0.548       0.392
alert_coverage          1.1697      0.037     31.729      0.000       1.097       1.242
n_atoms                 0.0013      0.001      0.941      0.348      -0.001       0.004
heteroatom_fraction    -0.3298      0.094     -3.526      0.001      -0.514      -0.146
aromatic_fraction      -0.0182      0.041     -0.441      0.660      -0.099       0.063
mean_degree             0.0177      0.104      0.170      0.866      -0.188       0.223
degree_spread           0.1066      0.084      1.271      0.205      -0.059       0.272
R^2 = 0.865

                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const              

## 5. Overlap between partitions

The family comparison treats fifteen scaffold partitions as repeated observations. They are
drawn from one compound pool, so the test sets they produce are not independent, and the
degree of dependence bears on how the accompanying tests should be read. The pairwise
overlap between test sets is measured here, together with the share of test scaffolds that
also appear in training under each partitioning scheme.

In [7]:
SCAFFOLD_SEEDS = list(range(42, 57))
RANDOM_SEEDS   = list(range(42, 57))

def test_sets(seeds, scheme):
    out = {}
    for sd in seeds:
        tr, va, te = (make_scaffold_split(df, sd) if scheme == "scaffold"
                      else make_random_split(df, sd))
        out[sd] = (set(te["canonical"]), set(map(safe_scaffold, tr["canonical"])),
                   set(map(safe_scaffold, te["canonical"])))
    return out

rows = []
for scheme, seeds in [("scaffold", SCAFFOLD_SEEDS), ("random", RANDOM_SEEDS)]:
    ts = test_sets(seeds, scheme)
    js = []
    for i, a in enumerate(seeds):
        for b in seeds[i+1:]:
            A, B = ts[a][0], ts[b][0]
            js.append(len(A & B) / len(A | B))
    leak = [100 * len(ts[sd][2] & ts[sd][1]) / max(len(ts[sd][2]), 1) for sd in seeds]
    rows.append({"scheme": scheme, "n_partitions": len(seeds),
                 "mean_pairwise_jaccard": float(np.mean(js)),
                 "min_jaccard": float(np.min(js)), "max_jaccard": float(np.max(js)),
                 "expected_if_independent": 0.1 / (2 - 0.1),
                 "mean_test_scaffolds_in_training_pct": float(np.mean(leak))})
part = pd.DataFrame(rows)
part.to_csv(f"{R}/table31_partition_overlap.csv", index=False)
print(part.round(4).to_string(index=False))
print()
print("Two independent 10% samples of the same pool would overlap at a Jaccard of about")
print("0.053. Values well above that indicate that the partitions repeat one another.")

  scheme  n_partitions  mean_pairwise_jaccard  min_jaccard  max_jaccard  expected_if_independent  mean_test_scaffolds_in_training_pct
scaffold            15                 0.6335       0.6098       0.6575                   0.0526                               0.0000
  random            15                 0.0528       0.0379       0.0669                   0.0526                              67.5876

Two independent 10% samples of the same pool would overlap at a Jaccard of about
0.053. Values well above that indicate that the partitions repeat one another.


## 6. Sensitivity to the alert library

Eighteen alerts are used. Removing each in turn shows whether the agreement rests on one
entry, and the per-alert coverage shows how much of the evaluation set each contributes.

In [8]:
base_scores = desc_scores("hetero_or_neighbour")
base = np.mean([lift(overlap(base_scores, tie_seed=2000 + j)) for j in range(50)])

rows = []
for name in TOXICOPHORES:
    keep = {k: v for k, v in TOX_PATTERNS.items() if k != name}
    ev, ch = [], []
    for e in EVAL:
        t = tox_atoms(e["mol"], keep)
        if not t or len(t) == e["n_atoms"]:
            continue
        ev.append({**e, "tox": t}); ch.append(len(t) / e["n_atoms"])
    ch = np.array(ch)
    if len(ev) < 30:
        rows.append({"alert_removed": name, "n_eval": len(ev), "lift": np.nan,
                     "molecules_matched": np.nan}); continue
    sc = []
    for e in ev:
        mol = e["mol"]; ats = list(mol.GetAtoms())
        sc.append(np.array([1.0 if (a.GetSymbol() in HETERO or
                   any(nb.GetSymbol() in HETERO for nb in a.GetNeighbors())) else 0.0
                   for a in ats]))
    v = np.mean([lift(overlap(sc, tie_seed=3000 + j, ev=ev, chance=ch), chance=ch)
                 for j in range(50)])
    matched = sum(1 for e in EVAL if e["mol"].HasSubstructMatch(TOX_PATTERNS[name]))
    rows.append({"alert_removed": name, "n_eval": len(ev), "lift": v,
                 "molecules_matched": matched})

alerts = pd.DataFrame(rows)
alerts["change_vs_full"] = alerts["lift"] - base
alerts = alerts.sort_values("molecules_matched", ascending=False)
alerts.to_csv(f"{R}/table32_alert_sensitivity.csv", index=False)
print(f"full library, structural ranking: {base:.4f}\n")
print(alerts.round(4).to_string(index=False))

full library, structural ranking: 1.0540

         alert_removed  n_eval   lift  molecules_matched  change_vs_full
   polycyclic_aromatic     174 1.7049                 97          0.6509
        aromatic_amine     208 1.0074                 50         -0.0467
               epoxide     215 0.9425                 43         -0.1116
        aromatic_nitro     216 1.0068                 32         -0.0473
      michael_acceptor     220 1.0358                 30         -0.0183
             aziridine     228 1.0330                 15         -0.0210
      aliphatic_halide     236 1.0517                 10         -0.0023
          azo_aromatic     236 1.0450                  8         -0.0090
      nitrogen_mustard     237 1.0474                  7         -0.0066
           nitrosamine     232 1.0494                  5         -0.0046
       alkyl_sulfonate     234 1.0435                  3         -0.0105
    hetero_hetero_bond     235 1.0477                  3         -0.0064
         

## 7. Summary

In [9]:
print("Structural rankings, strongest first")
for _, r in struct.head(6).iterrows():
    if r.ranking == "perfect ranking (ceiling)":
        continue
    print(f"  {r.ranking:24s} {r.lift_mean:.4f}  "
          f"[{r.lift_p2_5:.4f}, {r.lift_p97_5:.4f}]  "
          f"{r.pct_of_ceiling:5.1f}% of ceiling")
try:
    u = pd.read_csv(f"{R}/table12_untrained_null.csv")
    u = u[u.arch == "gin"]["mean"].iloc[0]
    print(f"\n  untrained GIN            {u:.4f}")
except Exception:
    pass
print("\nWritten to", R)
for f in ["table29_structural_baselines.csv", "table30_signal_regression.csv",
          "table31_partition_overlap.csv", "table32_alert_sensitivity.csv"]:
    print("  -", f)

import shutil
from google.colab import files
shutil.make_archive("/content/part7_results", "zip", R)
files.download("/content/part7_results.zip")

Structural rankings, strongest first
  degree                   1.2135  [1.1899, 1.2386]   23.4% of ceiling
  hetero_neighbours        1.1632  [1.1317, 1.1941]   17.8% of ceiling
  degree_deviation         1.1295  [1.1060, 1.1534]   14.2% of ceiling
  hetero_plus_degree       1.0922  [1.0724, 1.1128]   10.1% of ceiling
  abs_formal_charge        1.0669  [1.0244, 1.1109]    7.3% of ceiling

  untrained GIN            1.1156

Written to /content/drive/MyDrive/ames_pipeline/results
  - table29_structural_baselines.csv
  - table30_signal_regression.csv
  - table31_partition_overlap.csv
  - table32_alert_sensitivity.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>